# EECE 6544: Introduction to Machine Learning and Pattern Recognition

## Assignment 03 · WheelsBazaar Used-Car Price Prediction

Use this notebook to build a multivariable linear regression model that predicts the fair market price of a used car from its attributes. Keep the work organized into the sections below and replace each placeholder comment with your actual analysis, code, and short written interpretations.

## 1. Load and Inspect the Data

- Load `car details v4.csv`.
- Confirm the dataset shape.
- Verify that `Price` is the continuous target.
- Check the main categorical and numeric fields for missing values.
- Display `head()`, `shape`, `info()`, `describe()`, and missing-value counts.

In [36]:
import pandas as pd

# Load the used-car dataset
df = pd.read_csv('car details v4.csv')

print('Shape:', df.shape)
print('\nHead:')
print(df.head())
print('\nInfo:')
df.info()
print('\nDescribe:')
print(df.describe(include='all'))
print('\nMissing values:')
print(df.isnull().sum().sort_values(ascending=False))

Shape: (2059, 20)

Head:
            Make                            Model    Price  Year  Kilometer  \
0          Honda              Amaze 1.2 VX i-VTEC   505000  2017      87150   
1  Maruti Suzuki                  Swift DZire VDI   450000  2014      75000   
2        Hyundai             i10 Magna 1.2 Kappa2   220000  2011      67000   
3         Toyota                         Glanza G   799000  2019      37500   
4         Toyota  Innova 2.4 VX 7 STR [2016-2020]  1950000  2018      69000   

  Fuel Type Transmission   Location   Color   Owner Seller Type   Engine  \
0    Petrol       Manual       Pune    Grey   First   Corporate  1198 cc   
1    Diesel       Manual   Ludhiana   White  Second  Individual  1248 cc   
2    Petrol       Manual    Lucknow  Maroon   First  Individual  1197 cc   
3    Petrol       Manual  Mangalore     Red   First  Individual  1197 cc   
4    Diesel       Manual     Mumbai    Grey   First  Individual  2393 cc   

            Max Power              Max Torq

## 2. Prepare the Features

- Use all available car attributes except the target `Price`.
- Keep the numeric columns as they are already stored in numeric form in the CSV.
- Convert the non-numeric columns into numeric form with `pd.get_dummies(..., drop_first=True)`.
- Fit a simple Ridge regression model on the transformed data.

In [37]:
import numpy as np
from sklearn.linear_model import Ridge

# Use every feature except the target column.
X = df.drop(columns=['Price']).copy()
X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median(numeric_only=True))
y = np.log1p(df['Price'].copy())

print('Predictor shape:', X.shape)
print('Target shape:', y.shape)
print('\nPredictor dtypes:')
print(X.dtypes.value_counts())

Predictor shape: (2059, 1928)
Target shape: (2059,)

Predictor dtypes:
bool       1921
float64       5
int64         2
Name: count, dtype: int64


## 3. Split the Data

In [38]:
from sklearn.model_selection import train_test_split

# split the data into training and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Fit the Ridge Regression Model

In [39]:
# fit the Ridge model on the training data
model = Ridge(alpha=1.0)
model.fit(X_train, Y_train)

Ridge()

## 5. Report the Model Parameters

In [40]:
coefficients = pd.Series(model.coef_, index=X.columns)

print('Intercept:', model.intercept_)
print('Alpha:', model.alpha)
coefficients.sort_values(key=lambda s: s.abs(), ascending=False).head(15)

Intercept: -206.2752532709985
Alpha: 1.0


Model_E-Class E 200 Avantgarde    0.845737
Make_Porsche                      0.706470
Make_Land Rover                   0.571214
Model_Commuter HiAce 3.0 L        0.561700
Model_Alto VXI                   -0.477319
Model_LX SUV Petrol               0.476227
Model_Discovery 3.0 TDV6 HSE      0.459574
Model_XF R 5.0 V8 Supercharged    0.456308
Make_Maruti Suzuki               -0.453000
Make_Honda                       -0.449140
Make_MINI                         0.446643
Make_Tata                        -0.440661
Max Power_570 bhp @ 5250 rpm      0.438424
Make_Rolls-Royce                  0.438424
Engine_6592 cc                    0.438424
dtype: float64

## 6. Evaluate on the Test Set

In [41]:
# generate predictions on the test set
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

Y_pred_log = model.predict(X_test)
Y_pred = np.expm1(Y_pred_log)
Y_test_price = np.expm1(Y_test)
# calculate R2, MAE, and RMSE
r2 = r2_score(Y_test_price, Y_pred)
mae = mean_absolute_error(Y_test_price, Y_pred)
rmse = root_mean_squared_error(Y_test_price, Y_pred)
print(f'R2: {r2:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}')

R2: 0.5865, MAE: 330803.6565, RMSE: 1699653.9637


## 7. Interpret the Results

The numeric fields are used directly because they are already stored in numeric form in the dataset. The categorical fields are one-hot encoded with `pd.get_dummies(..., drop_first=True)`, and Ridge keeps the fit stable while using all available features.

Newer cars with lower kilometers and stronger engine specifications should generally receive higher predicted prices, while older cars with more usage should be priced lower.

## 8. Make a Prediction for a New Listing

Create a sample vehicle listing and use the fitted model to estimate its fair market price.

In [42]:
new_car = df.drop(columns=['Price']).iloc[[0]].copy()
new_car['Year'] = 2020
new_car['Kilometer'] = 32000
new_car['Fuel Type'] = 'Diesel'
new_car['Transmission'] = 'Manual'
new_car['Location'] = 'Bangalore'
new_car['Owner'] = 'First'
new_car['Seller Type'] = 'Individual'

new_car = pd.get_dummies(new_car, drop_first=True)
new_car = new_car.reindex(columns=X.columns, fill_value=0)

Y_pred_new_log = model.predict(new_car)
Y_pred_new = np.expm1(Y_pred_new_log)
print(f'Predicted market price for the new listing: {Y_pred_new[0]:,.0f}')

Predicted market price for the new listing: 1,198,885
